In [ ]:
import numpy as np
import pandas as pd

In [ ]:
"""
Input: 
    df : pd.DataFrame
    uses the us_combined_data.csv
Returns:
    pd.DataFrame
    A copy of the DataFrame with normalized types
"""
def validate_data(df):
    df = df.copy()
    
    df["season"] = df["season"].astype(int)
    df["date"] = pd.to_datetime(df["date"])
    df["league"] = df["league"].astype(str)
    df["team1"] = df["team1"].astype(str)
    df["team2"] = df["team2"].astype(str)
    df["score1"] = df["score1"].astype(int)
    df["score2"] = df["score2"].astype(int)
    df["result"] = df["result"].astype(int)
    return df

In [ ]:
"""
Input: 
    df (pd.DataFrame)
        uses the validated us_combined_data.csv

Returns:
    pd.DataFrame
        Has two row pers game, one per each team
        (league, season, team_id, points_for, points_against, result, is_home)
        result is from THIS team perspective: 1 = win, 0 = tie, -1 = loss
        is_home: 1 for home team row, 0 for away team row

How it works:
    The code splits each game into a home row and an away row.
    We keep "result" the same for the home row (home perspective).
    The sign of "result" is flipped for the away row (away perspective).
    The two rows are concatenated.
    This creates a dataset that is useful for season aggregation.
"""
def create_team_rows(df):

    home = pd.DataFrame({
        "league": df["league"],
        "season": df["season"],
        "team_id": df["team1"],
        "points_for": df["score1"],
        "points_against": df["score2"],
        "result": df["result"],
        "is_home": 1  
    })

    away = pd.DataFrame({
        "league": df["league"],
        "season": df["season"],
        "team_id": df["team2"],
        "points_for": df["score2"],
        "points_against": df["score1"],
        "result": -df["result"],   # flip: 1->-1, -1->1, 0->0
        "is_home": 0 
    })

    return pd.concat([home, away], ignore_index = True)


In [ ]:
"""
Input: 
    rows (pd.DataFrame)
        uses the output of 'create_team_rows' function

Returns:
    pd.DataFrame
        End of season standinger per league, season, team_id

How it works:
    The code caluclates the number of win/losses/ties from 'result'.
    We group by (league, season, team_id) and then aggregate the season totals.
    The win percentage and point differential are then calculated.
    We sort within each leageu by win_pct (DESC), point_diff (DESC), team_id (ASC).
    Ranks are then assigned, 1 being the best.
"""
def get_standing(rows):

    team_rows = rows.copy()
    team_rows["wins"] = (team_rows["result"] == 1).astype(int)
    team_rows["losses"] = (team_rows["result"] == -1).astype(int)
    team_rows["ties"] = (team_rows["result"] == 0).astype(int)

    standings = team_rows.groupby(["league", "season", "team_id"], as_index=False).agg({
        "result": "count",
        "wins": "sum",
        "losses": "sum", 
        "ties": "sum",
        "points_for": "sum",
        "points_against": "sum"
    }).rename(columns={"result": "games_played"})

    standings["win_pct"] = (standings["wins"] + 0.5 * standings["ties"]) / standings["games_played"]
    standings["point_diff"] = standings["points_for"] - standings["points_against"]
    standings["win_pct"] = standings["win_pct"].round(3)

    standings = standings.sort_values(
        ["league", "season", "win_pct", "point_diff", "team_id"],
        ascending=[True, True, False, False, True]
    )
    standings["rank"] = standings.groupby(["league", "season"]).cumcount() + 1

    columns = ["league", "season", "team_id", "games_played", "wins", "losses", 
               "ties", "win_pct", "points_for", "points_against", "point_diff", "rank"]
    
    return standings[columns]

In [ ]:

"""
Input: 
    standings (pd.DataFrame)
        Output of 'get_standing()'
    output (str)
        The directory of where the CSV will be written
    filename (str)
        Name of the CSV file to create

Returns:
    str
        The full path to the written CSV

How it works:
    It sorts by league, season, and rank for readability.
    Then writes a csv to output with filename.
    Print a short summary of where it was exported and the total number of row.
"""
def get_rankings(standings, output="../csv/end_of_season", filename="end_of_season_us.csv"):
    path = f"{output}/{filename}"
    standings_sorted = standings.sort_values(["league", "season", "rank"])
    standings_sorted.to_csv(path, index=False)
    print(f"Exported: {path}")

    print(f"Total rows: {len(standings_sorted)}")
    return path


In [ ]:
# run to generate end_of_season_us_pure_skill.csv
all_data = pd.read_csv("../../data/us_leagues/csv/game_by_game/pure_skill/us_combined_pure_skill.csv")
clean = validate_data(all_data)
team_rows = create_team_rows(clean)
standings = get_standing(team_rows)
output_path = get_rankings(standings, output="../../data/us_leagues/csv/end_of_season/pure_skill", filename="end_of_season_us_pure_skill.csv")

In [ ]:
# run to generate end_of_season_us.csv
all_data = pd.read_csv("../csv/us_combined_data.csv")
clean = validate_data(all_data)
team_rows = create_team_rows(clean)
standings = get_standing(team_rows)
output_path = get_rankings(standings, output="../csv/end_of_season", filename="end_of_season_us.csv")

In [ ]:
# run to generate end_of_season_us_coin_flip_0.5.csv
all_data = pd.read_csv("../csv/us_combined_data_coin_flip_0.5.csv")
clean = validate_data(all_data)
team_rows = create_team_rows(clean)
standings = get_standing(team_rows)
output_path = get_rankings(standings,output="../csv/end_of_season",filename="end_of_season_us_coin_flip_0.5.csv")

In [ ]:
# run to generate end_of_season_us_coin_flip_0.7.csv
all_data = pd.read_csv("../csv/us_combined_data_coin_flip_0.7.csv")
clean = validate_data(all_data)
team_rows = create_team_rows(clean)
standings = get_standing(team_rows)
output_path = get_rankings(standings,output="../csv/end_of_season",filename="end_of_season_us_coin_flip_0.7.csv")

In [ ]:
# run to generate mlb_standings.csv
mlb_data = pd.read_csv("../csv/mlb_data.csv")
mlb_clean = validate_data(mlb_data)
team_rows = create_team_rows(mlb_clean, league="MLB")
standings = get_standing(team_rows)
output_path = get_rankings(standings, output="../csv", filename = "end_of_season_mlb.csv")

In [ ]:
# run to generate nba_standings.csv
nba_data = pd.read_csv("../csv/nba_data.csv")
nba_clean = validate_data(nba_data)
team_rows = create_team_rows(nba_clean, league="NBA")
standings = get_standing(team_rows)
output_path = get_rankings(standings, output="../csv", filename = "end_of_season_nba.csv")